In [2]:

from pathlib import Path
import sys
import json
import pandas as pd

# Define the directory and file paths
PARQUET_DIR = Path("data/inbox/")
files = PARQUET_DIR.glob("*.parquet") # Only files with extension .parquet

state_dir = Path("data/output")
state_file_path = state_dir / "state.json"


In [3]:
# Kontrollib, kas json on olemas, kui ei ole ss loob tühja faili, et lõpus sinna kirjutada

def check_or_create_state():

    # 1. Create the directory if it doesn't exist
    state_dir.mkdir(exist_ok=True)

    # 2. Check if file exists
    if state_file_path.exists():
        return "state.json already exists"
    
    # 3. Else create a new empty state.json file
    else:
        state_file_path.touch()
        return "state.json created successfully"

# Execute and print the result
print(check_or_create_state())

state.json already exists


# 2. TRANSFORMATIONS
 - parse and cast  types correctly (timestamps, numeric fields)
 - Apply data cleaning rules (invalid values, null handling). Document the rules.
 - Deduplicate records using a defined key. Document the key.

In [ ]:
# --- Single-file transform ---
def transform_file(df: pd.DataFrame) -> pd.DataFrame:
    # Parse timestamps
    date_cols = [col for col in df.columns if 'date' in col.lower() or 'time' in col.lower()]
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')
    
    # Parse numeric columns
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Data cleaning
    if date_cols:
        df = df.dropna(subset=date_cols)  # eemalda read, kus date/time on vigane
    for col in numeric_cols:
        df = df[df[col] >= 0]  # eemalda negatiivsed väärtused

    # Deduplicate
    dedup_key = ['id'] if 'id' in df.columns else []
    if date_cols:
        dedup_key.append(date_cols[0])  # kasuta esimest kuupäeva/aja veergu dedup key-s
    if dedup_key:
        df = df.drop_duplicates(subset=dedup_key)

    return df

# --- Kontrolli iga faili eraldi lõpus ---
# 1. Loe state.json
if state_file_path.exists() and state_file_path.stat().st_size > 0:
    with open(state_file_path, "r") as f:
        processed_files = json.load(f).get("processed_files", [])
else:
    processed_files = []

# 2. Käi läbi iga fail inboxist
for file_path in files:
    file_name = file_path.name

    if file_name in processed_files:
        print(f"{file_name}: You already have outputs from this file")
    else:
        print(f"Running transformation for: {file_name}")

        # Loe fail
        df = pd.read_parquet(file_path)

        # Transform
        df_transformed = transform_file(df)

        # Salvesta output
        OUTPUT_DIR = Path("data/outbox")
        OUTPUT_DIR.mkdir(exist_ok=True)
        output_file = OUTPUT_DIR / f"{file_name.replace('.parquet','')}_output.parquet"
        df_transformed.to_parquet(output_file, index=False)

        print(f"Output saved to {output_file}")

        # Uuenda state.json
        processed_files.append(file_name)
        with open(state_file_path, "w") as f:
            json.dump({"processed_files": processed_files}, f)

taxi_zone_lookup.parquet: You already have outputs from this file
yellow_tripdata_2025-01.parquet: You already have outputs from this file
Running transformation for: yellow_tripdata_2025-02.parquet


# 3. Enrichment
 - With taxi_zone_lookup for pickup and dropoff zones.

**Scenario:**
    Support a config parameter months N (default: all) that limits the enriched output to trips from the last N calendar months relative to the latest pickup date in the data. The manifest still tracks all ingested files. README must show how to invoke it.

In [ ]:
def enrich():
    ...

# 4. Output
 - Write a single output dataset to data/outbox/trips_enriched.parquet.
 - The output must contain all processed data (including previous runs).

## Required output fields
 - pickup and dropoff timestamps 
 - pickup and dropoff LocationID 
 - pickup and dropoff zone name (from lookup) 
 - passenger_count, trip_distance 
 - derived: trip_duration_minutes, pickup_date 
 - metadata: source_file, ingested_at

In [ ]:
# loeb json faili, võrdleb inbox failidega

# 1. Read "state.json" file
if state_file_path.exists() and state_file_path.stat().st_size > 0:
    with open(state_file_path, "r") as f:
        processed_files = json.load(f)
else:
    processed_files = [] # Fallback if file is empty or missing

# 2. Check every file in "files" separately
for file_path in files:
    file_name = file_path.name
    
    if file_name in processed_files:
        # 3. If it appears
        print(f"{file_name}: You already have outputs from this file")
    else:
        # 4. Else run transformation, enrich, write to json and give an output
        print(f"Running transformation function for: {file_name}")
        
        # Placeholders:
        # transform(file_path)
        # enrich(file_path)
        # write to json
        # output
